# OpenPlaque — LAD Takeoff Root-Directed Alternatives v2

Start only from the previously validated proximal LAD (~57 mm backbone). Generate a small set of distinct root-directed proximal alternatives, recenter every step on an RCA-calibrated source-resolution lumen component, and compare them by lumen QC versus progress toward the TotalSegmentator aorta. A short local branch probe is used only to look for bifurcation evidence; it does **not** trace the LCX. Research use only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse controls — True + valid cache reuses it; False forces recomputation and overwrite.
REUSE_SOURCE_CT = True
REUSE_VALIDATED_LAD = True
REUSE_AORTA_CONSTRAINT = True
REUSE_ALTERNATIVES = True
REUSE_BRANCH_PROBE = True
REUSE_FIGURES = True
REUSE_REPORT = True


## Step 3 — Install dependencies


In [ ]:
%pip -q install pydicom SimpleITK scipy matplotlib pandas psutil 'pylibjpeg>=2.0' 'pylibjpeg-libjpeg>=2.1'
print('Dependencies ready, including JPEG Lossless decoding.')


## Step 4 — Load this fresh branch


In [ ]:
import os, sys, subprocess, shutil
REPO='/content/OpenPlaque'
BRANCH='lad-takeoff-root-alternatives-from-main'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',REPO],check=True)
sys.path.insert(0, os.path.join(REPO,'src'))
print('Loaded', BRANCH)


## Step 5 — Initialize workflow and inspect caches


In [ ]:
from openplaque.lad_takeoff_root_alternatives_v2 import LADTakeoffRootAlternativesWorkflow
reuse={
 'source_ct':REUSE_SOURCE_CT,
 'validated_lad':REUSE_VALIDATED_LAD,
 'aorta_constraint':REUSE_AORTA_CONSTRAINT,
 'alternatives':REUSE_ALTERNATIVES,
 'branch_probe':REUSE_BRANCH_PROBE,
 'figures':REUSE_FIGURES,
 'report':REUSE_REPORT,
}
wf=LADTakeoffRootAlternativesWorkflow(reuse=reuse)
display(wf.cache_status())


## Step 6 — Load source CCTA and revalidate the existing LAD backbone

This does **not** search for the LAD again. It loads the prior `LAD_Proximal_Recenter_v1/combined_lad_centerline.csv`, reuses the validated RCA calibration, and checks the full backbone in true source-resolution orthogonal planes.


In [ ]:
wf.load_source_ct()
lad_summary=wf.validate_lad_backbone()
print(lad_summary)
assert lad_summary['accepted'], 'Previously validated LAD failed the safety revalidation gate.'


## Step 7 — Build a fresh local TotalSegmentator aorta constraint

The TotalSegmentator result is used only for source-space aortic distance/exclusion and root-direction guidance. It does not identify the LAD.


In [ ]:
wf.build_aorta_constraint()
print('Aorta constraint ready.')


## Step 8 — Generate and compare root-directed proximal alternatives

Every step is recentered on a source-resolution coronary-sized lumen component. The search keeps several distinct alternatives instead of forcing one trajectory, so lumen quality can be compared directly against progress toward the aortic root.


In [ ]:
alts=wf.search_alternatives(max_extension_mm=28.0, beam_width=14, n_alternatives=4)
display(alts)


## Step 9 — Probe locally for LAD takeoff/bifurcation evidence

This is deliberately short-range. It asks whether a point along a candidate has multiple non-LAD coronary-like directions over a few millimeters. It does **not** trace or label an LCX centerline.


In [ ]:
takeoff=wf.probe_takeoff()
print(takeoff)
if wf.branch_probe is not None:
    display(wf.branch_probe.sort_values('branch_evidence_score',ascending=False).head(12))


## Step 10 — Generate QC figures


In [ ]:
figs=wf.plot_qc()
for f in figs: print(f)


## Step 11 — Package report back to Drive


In [ ]:
z=wf.package()
print('REPORT BACK:', z)
print('Expected filename: OPENPLAQUE_LAD_TAKEOFF_ROOT_ALTERNATIVES_REPORT_BACK.zip')
